In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

In [ ]:
from pathlib import Path
import os

def _find_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "README.md").exists() and (candidate / "notebooks").is_dir():
            return candidate
    return current

PROJECT_ROOT = _find_project_root()
DATA_ROOT = Path(os.environ.get("GANGA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("GANGA_OUTPUT_ROOT", PROJECT_ROOT / "outputs")).expanduser()

PROJECT_DIR = PROJECT_ROOT
OUTPUT_DIR = OUTPUT_ROOT / "cpcb"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ML_DATASET = OUTPUT_DIR / "ML_Dataset_2015_2025.csv"


In [ ]:
# dataset load

ml = pd.read_csv(ML_DATASET)

print("Shape :", ml.shape)

ml.head()

In [ ]:
#basic inspection
print()

print("Rows :", len(ml))
print("Columns :", len(ml.columns))

print()

print("Stations :", ml["Station_Name"].nunique())

print()

print("Years :")

print(sorted(ml["Year"].unique()))

In [ ]:
# ==========================================================
# INPUT FEATURES
# ==========================================================

FEATURES = [
    "Blue",
    "Green",
    "Red",
    "VIIRS",
    "LULC"
]

print("Input Features")

FEATURES

In [ ]:
# ==========================================================
# TARGET PARAMETERS
# ==========================================================

TARGETS = [
    "_do",
    "bod",
    "cod",
    "ph_gen",
    "ec_gen",
    "temp"
]

print("Target Parameters")

TARGETS

In [ ]:
# Missing values

missing = ml[FEATURES + TARGETS].isna().sum()

missing

In [ ]:
summary = pd.DataFrame({

    "Missing": ml[FEATURES + TARGETS].isna().sum(),

    "Available": ml[FEATURES + TARGETS].notna().sum(),

    "DataType": ml[FEATURES + TARGETS].dtypes

})

summary

Outlier Inspection

In [ ]:
ml[FEATURES + TARGETS].describe().T

Generic Data Cleaning Function

In [ ]:
def prepare_dataset(df, target):

    columns = FEATURES + [target]

    data = df[columns].copy()

    # Remove rows where target is missing

    data = data.dropna(subset=[target])

    # Fill missing feature values using median

    for feature in FEATURES:

        data[feature] = data[feature].fillna(
            data[feature].median()
        )

    return data

In [ ]:
test = prepare_dataset(
    ml,
    "_do"
)

print(test.shape)

test.head()

Train/Test Split

In [ ]:
def split_dataset(data, target):

    X = data[FEATURES]

    y = data[target]

    return train_test_split(

        X,

        y,

        test_size=0.2,

        random_state=42

    )

In [ ]:
dataset = prepare_dataset(
    ml,
    "_do"
)

X_train, X_test, y_train, y_test = split_dataset(
    dataset,
    "_do"
)

print(X_train.shape)

print(X_test.shape)

Reusable Random Forest Training Engine

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
def train_random_forest(target):

    print("=" * 70)
    print(f"Training Model for: {target}")
    print("=" * 70)

    # -----------------------
    # Prepare Dataset
    # -----------------------

    dataset = prepare_dataset(
        ml,
        target
    )

    X_train, X_test, y_train, y_test = split_dataset(
        dataset,
        target
    )

    # -----------------------
    # Model
    # -----------------------

    model = RandomForestRegressor(

        n_estimators=300,

        random_state=42,

        n_jobs=-1

    )

    model.fit(
        X_train,
        y_train
    )

    # -----------------------
    # Prediction
    # -----------------------

    prediction = model.predict(
        X_test
    )

    # -----------------------
    # Metrics
    # -----------------------

    mae = mean_absolute_error(
        y_test,
        prediction
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            prediction
        )
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    # -----------------------
    # Feature Importance
    # -----------------------

    importance = pd.DataFrame({

        "Feature": FEATURES,

        "Importance": model.feature_importances_

    }).sort_values(
        "Importance",
        ascending=False
    )

    return {

        "model": model,

        "importance": importance,

        "prediction": prediction,

        "y_test": y_test,

        "metrics": {

            "MAE": mae,

            "RMSE": rmse,

            "R2": r2

        }

    }

In [ ]:
do_model = train_random_forest("_do")

In [ ]:
do_model["importance"]

In [ ]:
results = []

for target in TARGETS:

    output = train_random_forest(target)

    results.append({

        "Target": target,

        "MAE": output["metrics"]["MAE"],

        "RMSE": output["metrics"]["RMSE"],

        "R2": output["metrics"]["R2"]

    })

results = pd.DataFrame(results)

results

moved to nrml ML

Dependency installation is managed by the repository-level `requirements.txt`. Run `pip install -r requirements.txt` from the repository root instead of installing packages inside this notebook.

Dependency installation is managed by the repository-level `requirements.txt`. Run `pip install -r requirements.txt` from the repository root instead of installing packages inside this notebook.

Dependency installation is managed by the repository-level `requirements.txt`. Run `pip install -r requirements.txt` from the repository root instead of installing packages inside this notebook.

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor


In [ ]:
def train_model(model, model_name, target):

    # Prepare Dataset
    dataset = prepare_dataset(ml, target)

    # Split Dataset
    X_train, X_test, y_train, y_test = split_dataset(dataset, target)

    # Train Model
    model.fit(X_train, y_train)

    # Prediction
    prediction = model.predict(X_test)

    plot_actual_vs_predicted(
    y_test,
    prediction,
    target,
    model_name
)

    plot_feature_importance(
    model,
    target,
    model_name
)

# Metrics
    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    r2 = r2_score(y_test, prediction)

    return {
        "Model": model_name,
        "Target": target,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

models dictionary

In [ ]:
models = {

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        random_state=42,
        objective="reg:squarederror"
    ),

    "LightGBM": LGBMRegressor(
        n_estimators=300,
        random_state=42
    ),

    "CatBoost": CatBoostRegressor(
        iterations=300,
        random_state=42,
        verbose=False
    )

}

In [ ]:
results = []

for model_name, model in models.items():

    print("\n" + "="*80)
    print(f"Running {model_name}")
    print("="*80)

    for target in TARGETS:

        output = train_model(
            model,
            model_name,
            target
        )

        results.append(output)

In [ ]:
results = pd.DataFrame(results)

results

In [ ]:
results.sort_values(
    by="R2",
    ascending=False
)

In [ ]:
best_models = (
    results.loc[
        results.groupby("Target")["R2"].idxmax()
    ]
)

best_models

actual vs predicted plot

In [ ]:
import matplotlib.pyplot as plt

def plot_actual_vs_predicted(y_test, prediction, target, model_name):

    plt.figure(figsize=(6,6))

    plt.scatter(y_test, prediction, alpha=0.6)

    min_val = min(y_test.min(), prediction.min())
    max_val = max(y_test.max(), prediction.max())

    plt.plot(
        [min_val, max_val],
        [min_val, max_val],
        'r--'
    )

    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{model_name} - {target}")

    plt.grid(True)

    plt.show()

In [ ]:
import matplotlib.pyplot as plt

for target in TARGETS:

    temp = results[results["Target"] == target]

    plt.figure(figsize=(7,4))

    plt.bar(
        temp["Model"],
        temp["R2"]
    )

    plt.title(f"Model Comparison for {target}")

    plt.ylabel("R² Score")

    plt.xticks(rotation=20)

    plt.grid(axis="y")

    plt.show()

In [ ]:
results.to_csv(
    OUTPUT_DIR / "Model_Performance.csv",
    index=False
)

print("Model Performance Saved")